# ORC Basics — 04: Predicate Pushdown

ORC's predicate pushdown uses 3 levels of statistics.
Bloom filters add a fourth layer for equality predicates.

Topics: row-level stats, bloom filter pushdown, supported predicates, explain().


In [ ]:
import os, time, pathlib, shutil, random, datetime, subprocess, glob as glib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/orc_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('orc-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version}")

## Step 1 — Setup: Sorted ORC for Best Statistics



In [ ]:

ORC_PATH = f"{DATA_DIR}/pushdown_test.orc"
import random, datetime
random.seed(42); N=300_000
rows=[]
base=datetime.date(2023,1,1)
CATS=["Electronics","Clothing","Books","Food","Sports","Furniture"]
CTRS=["US","UK","DE","FR","JP","CA"]
for i in range(N):
    d=base+datetime.timedelta(days=random.randint(0,365))
    rows.append((i+1,random.randint(1,10000),random.choice(CATS),random.choice(CTRS),round(random.uniform(5,1000),2),d))
df_src = spark.createDataFrame(rows,["order_id","customer_id","category","country","revenue","order_date"])
df_src.orderBy("category","country","revenue") \
      .write.mode("overwrite") \
      .option("orc.bloom.filter.columns","category,country,status") \
      .orc(ORC_PATH)
print(f"Written {N:,} rows sorted → good statistics")


## Step 2 — Verify Pushdown with explain()



In [ ]:

# Equality filter — bloom filter + min/max
df_eq = spark.read.orc(ORC_PATH).filter(F.col("category")=="Electronics")
print("Equality filter plan:")
df_eq.explain(mode="formatted")
print("Look for PushedFilters in OrcScan")


## Step 3 — Range Filter (min/max statistics)



In [ ]:

# Range filter uses min/max stats to skip stripes
df_range = spark.read.orc(ORC_PATH).filter(F.col("revenue").between(800, 1000))
print("Range filter plan:")
df_range.explain(mode="formatted")

runs=3
tf,tr=[],[]
for _ in range(runs):
    t0=time.time(); spark.read.orc(ORC_PATH).agg(F.sum("revenue")).collect(); tf.append(time.time()-t0)
    t0=time.time(); spark.read.orc(ORC_PATH).filter(F.col("revenue")>900).agg(F.sum("revenue")).collect(); tr.append(time.time()-t0)
print(f"\nFull scan         : {sorted(tf)[1]:.3f}s")
print(f"revenue > 900 scan: {sorted(tr)[1]:.3f}s  (stripe/row-index skipping)")


## Step 4 — What Does NOT Push Down



In [ ]:

tests = [
    ("EqualTo",    "✅", lambda: spark.read.orc(ORC_PATH).filter(F.col("category")=="Electronics")),
    ("GreaterThan","✅", lambda: spark.read.orc(ORC_PATH).filter(F.col("revenue")>500)),
    ("In",         "✅", lambda: spark.read.orc(ORC_PATH).filter(F.col("category").isin("Electronics","Books"))),
    ("IsNull",     "✅", lambda: spark.read.orc(ORC_PATH).filter(F.col("revenue").isNull())),
    ("Contains",   "❌", lambda: spark.read.orc(ORC_PATH).filter(F.col("category").contains("Elect"))),
    ("UDF",        "❌", lambda: spark.read.orc(ORC_PATH).filter(F.udf(lambda x: x=="Electronics","boolean")(F.col("category")))),
]
print(f"{'Predicate':<15} {'Pushed?':>8}")
print("-"*28)
for label,expected,fn in tests:
    plan=fn()._jdf.queryExecution().executedPlan().toString()
    pushed="✅ YES" if "PushedFilters" in plan and "[]" not in plan.split("PushedFilters")[1][:50] else "❌ NO"
    print(f"  {label:<13} {pushed:>8}  (expected {expected})")


## Summary



In [ ]:

print("""
ORC predicate pushdown:
  ✅ EqualTo, LessThan, GreaterThan, LessThanOrEqual, GreaterThanOrEqual
  ✅ In (equality list)
  ✅ IsNull, IsNotNull
  ✅ And, Or combinations
  ❌ Contains / LIKE '%...'
  ❌ Python UDFs
  ❌ Derived/computed columns

Bloom filters add a FOURTH level:
  File stats → Stripe stats → Row index stats → Bloom filter → Row data
  Best for: equality filters on high-cardinality string/int columns
""")
